In [ ]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Defining Asset Operational Limits over User-Defined Time Periods

This example demonstrates how to impose **periodic minimum and maximum energy output constraints** on a Combined Cycle Gas Turbine (CCGT) generator using PyPSA's custom constraint mechanism.
This example focusses on monthly limits but the approach would work for limits set over any user-defined periods, including non-uniform ones (e.g, daily, weekly, every reporting period, etc.).

It uses a 3-hourly resolved variant of the single-node network developed in the [single-node capacity expansion example](capacity-expansion-planning-single-node.ipynb), replacing the load-shedding generator with a combined-cycle gas turbine (CCGT). 

In practice, monthly CCGT constraints arise from:
- contractual minimum-offtake agreements (lower bound),
- fuel-supply or CO₂-permit limits (upper bound).

Our operational limits will be added via the `extra_functionality` callback in `n.optimize()`.

In [ ]:
import calendar

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import xarray as xr

First, we will create the definition for our new CCGT generator, and use it in our network in place of load shedding.

We attach the monthly limits to the network as custom time-dependent attributes (`p_min_periodic`, `p_max_periodic`), with values only on the first snapshot of each month (all other snapshots are `NaN`). 
PyPSA stores any extra keyword arguments passed to `n.add()` as component attributes, so these custom attributes persist when exporting the network to file / importing the network from file.


If we wanted to apply a different period for our limit, we would update the attribute to have the limit data applied at the first snapshot of each period, e.g.:

```py
daily_limit = pd.Series([10, 15, ...], index=pd.to_datetime(["2019-01-01", "2019-01-02", ...]))
# We have to expand the data out to all snapshots to be able to store it as a PyPSA network attribute.
# All snapshots that aren't the first of the day will be filled with NaN.
daily_limit_all_snapshots = daily_limit.reindex(n.snapshots)
n.add(
    "Generator",
    "CCGT",
    ...
    p_max_periodic=daily_limit_all_snapshots,
)
```

In [ ]:
def get_p_monthly(network: pypsa.Network, output_mw: float) -> pd.Series:
    """Define a monthly output limit for dispatch, set for the first snapshot of each month."""
    sns = network.snapshots
    weights = network.snapshot_weightings["generators"]
    return pd.Series(output_mw * weights, index=sns).resample("MS").sum().reindex(sns)


n = pypsa.examples.model_energy()

ccgt_p_nom = 2000  # MW – fixed installed capacity
min_fraction = 0.15  # lower bound: 15 % of maximum possible monthly output
max_fraction = 0.40  # upper bound: 40 % of maximum possible monthly output
p_min_monthly = get_p_monthly(n, min_fraction * ccgt_p_nom)
p_max_monthly = get_p_monthly(n, max_fraction * ccgt_p_nom)

# Replace load shedding with a fixed-capacity CCGT
n.remove("Generator", "load shedding")

n.add(
    "Generator",
    "CCGT",
    bus="electricity",
    carrier="CCGT",
    p_nom=ccgt_p_nom,
    marginal_cost=60,  # €/MWh
    p_nom_extendable=False,
    p_min_periodic=p_min_monthly,
    p_max_periodic=p_max_monthly,
)
n.add("Carrier", "CCGT")

monthly_bounds = (
    pd.concat([p_min_monthly, p_max_monthly], axis=1, keys=["p_min", "p_max"])
    .groupby(n.snapshots.month.rename("month"))
    .sum()
)
display(monthly_bounds)

## Apply Custom Constraints

The `extra_functionality` callback receives the network `n` and the active snapshot index `sns`.
Inside the callback we:

1. Retrieve the CCGT active-power variable from the Linopy model (`n.model`).
2. Sum the variable over all (weighted) snapshots in a period (here, months):
   $$
   E_m = \sum_{t \in \text{month } m} w_t \cdot p_{\text{CCGT},t}
   $$
   where $w_t$ is the snapshot weighting (1 h for hourly data).

3. Add two constraints per month:
   $$
   E_m \geq E_m^{\min}, \quad E_m \leq E_m^{\max}
   $$

In [ ]:
def get_grouper(limit_attr: xr.DataArray) -> xr.DataArray:
    """Return a grouper for snapshots in the same period based on the NaN values between each non-NaN value in the limit attribute."""
    return limit_attr.notnull().cumsum("snapshot").rename("limit_period")


def period_ccgt_output_constraints(n: pypsa.Network, sns: pd.Index) -> None:
    """Add lower and upper energy-output constraints for the CCGT dispatch per period.

    Parameters
    ----------
    n : pypsa.Network
    sns : pd.Index
        Active snapshots passed by PyPSA during optimisation.
    """
    m = n.model
    selector = {"name": "CCGT", "snapshot": sns}
    p_ccgt = m.variables["Generator-p"].sel(**selector)
    weights = n.snapshot_weightings["generators"].loc[sns]
    for limit, sign in {"max": "<=", "min": ">="}.items():
        limit_attr = n.components["Generator"].da[f"p_{limit}_periodic"].sel(**selector)
        grouper = get_grouper(limit_attr)
        p_period = (p_ccgt * weights).groupby(grouper).sum()
        p_lim_period = limit_attr.groupby(grouper).sum()
        m.add_constraints(
            lhs=p_period, sign=sign, rhs=p_lim_period, name=f"CCGT-period-{limit}"
        )

## Run Optimization

We solve the network **twice**:

1. **Baseline** – no custom constraints, purely cost-optimal dispatch.
2. **Constrained** – the `extra_functionality` callback enforces monthly CCGT bounds.

Both runs use the default HiGHS solver.

In [ ]:
# --- Baseline: no custom constraints ---
n_base = n.copy()
n_base.optimize(include_objective_constant=False)
print("Baseline solved")

In [ ]:
# --- Constrained: with monthly CCGT output limits ---
n_constrained = n.copy()
n_constrained.optimize(
    extra_functionality=period_ccgt_output_constraints,
    include_objective_constant=False,
)
print("Constrained solve completed")

We can view the constraints that has been built by querying linopy directy:

In [ ]:
display(n_constrained.model.constraints["CCGT-period-min"])
display(n_constrained.model.constraints["CCGT-period-max"])

## Visualize Results

### Monthly CCGT output vs. constraints

The bar chart below shows the monthly CCGT energy output for both the baseline and the constrained run alongside the prescribed lower and upper bounds.
In the constrained run the bars should always fall within the shaded region.

In [ ]:
def periodic_energy_balance(
    network: pypsa.Network | pypsa.NetworkCollection, grouper: pd.Series
) -> pd.Series:
    """Return periodic energy output [MWh] for a solved network.

    The `grouper` defines which snapshots belong to the same period, e.g. by month or by season.
    """
    weights = network.snapshot_weightings.generators
    energy_balance = network.statistics.energy_balance(
        groupby_time=False, bus_carrier="electricity"
    ).droplevel(["component", "bus_carrier"])
    energy_balance_period = (energy_balance * weights).T.groupby(grouper).sum()
    return energy_balance_period


def discharge_capacity(network: pypsa.Network | pypsa.NetworkCollection) -> pd.Series:
    """Return discharge capacity [MW] for a solved network."""
    generators = network.generators.p_nom_opt
    storage_units = network.storage_units.p_nom_opt
    capacity = pd.concat([generators, storage_units])
    return capacity


nc = pypsa.NetworkCollection({"Baseline": n_base, "Constrained": n_constrained})
grouper = get_grouper(
    n_constrained.components["Generator"].da["p_min_periodic"].sel(name="CCGT")
).to_series()

In [ ]:
monthly_energy_balance = periodic_energy_balance(nc, grouper).xs(
    "CCGT", level="carrier", axis=1
)

month_labels = [calendar.month_abbr[m] for m in monthly_bounds.index]
df_plot = (
    monthly_energy_balance.stack().to_frame("Monthly CCGT dispatch (MWh)").reset_index()
)
df_plot["Month"] = df_plot["limit_period"].apply(lambda m: calendar.month_abbr[m])
fig = px.bar(
    df_plot,
    x="Month",
    y="Monthly CCGT dispatch (MWh)",
    color="network",
    barmode="group",
)

# Overlay the monthly bounds to verify that CCGT dispatch is within the limits
common_attrs = {
    "x": month_labels,
    "mode": "lines",
    "legendgroup": "bound",
    "line": {"width": 1, "shape": "hvh", "color": "black"},
}
fig.add_trace(go.Scatter(y=monthly_bounds["p_min"], showlegend=False, **common_attrs))
fig.add_trace(
    go.Scatter(
        y=monthly_bounds["p_max"],
        name="Feasible constrained range",
        fill="tonexty",
        fillcolor="rgba(169, 169, 169, 0.3)",
        **common_attrs,
    )
)

### Monthly generation breakdown

Since the segmented network uses non-uniform snapshot durations, we aggregate by computing the **weighted energy** (`p × snapshot_weighting`) per technology per month. This gives total-generation [MWh] for each month and shows how the constraints redistribute generation between wind/solar, hydrogen, and the CCGT.


In [ ]:
capacity = discharge_capacity(nc)
df_plot = capacity.to_frame("Capacity (MW)").reset_index()
fig_capacity = px.bar(
    df_plot,
    x="name",
    y="Capacity (MW)",
    color="network",
    barmode="group",
)
fig_capacity.update_layout(
    xaxis_title="Generator / Storage unit", yaxis_title="Installed capacity (MW)"
)

### System cost comparison

The monthly energy bounds force the CCGT away from its cost-optimal dispatch level.
We compare the total system operating cost for both runs.

In [ ]:
with pd.option_context("display.float_format", "{:,.0f} M€".format):
    capex = nc.statistics.capex().groupby("network").sum() / 1e6
    opex = nc.statistics.opex().groupby("network").sum() / 1e6
    total = capex + opex
    cost_df = pd.concat([capex, opex, total], axis=1, keys=["Capex", "Opex", "Total"]).T
    cost_df["Cost delta"] = (
        cost_df["Constrained"]
        .subtract(cost_df["Baseline"])
        .div(cost_df["Baseline"])
        .mul(100)
        .round(1)
        .astype(str)
        + " %"
    )
    display(cost_df)